# Task 1: Multi-Class Benchmark Classification Pipeline
## Datheon — Machine Learning & Data Science Internship

**Prepared by:** Ritesh Kumar Singh | CSE (Data Science) Student

### Objective
Compare Logistic Regression and Decision Tree classifiers on the Iris dataset using a stratified 80/20 train-test split. Evaluate performance with accuracy, precision, recall, F1-score and confusion matrices.

### Dataset and provenance
The Datheon task page links to the Iris CSV hosted in the `mwaskom/seaborn-data` GitHub repository:
- Dataset: https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv
- Reference page: https://www.datheon.in/training/catalog/machine-learning-and-data-science-internship

To keep this notebook reproducible without a network connection, the code below uses scikit-learn's bundled Iris dataset, which contains the same standard 150-row, four-feature, three-class benchmark. The feature names are converted to match the CSV's column names.

### Methodological choices
- Use stratification so class proportions are preserved in train and test sets.
- Fit feature scaling only on the training data via a scikit-learn `Pipeline`, preventing data leakage.
- Use a fixed random seed for reproducibility.
- Treat the test set as a final evaluation set, not as a tuning set.


## 1. Import libraries and set reproducibility

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


## 2. Load and inspect the dataset

In [2]:
iris = load_iris(as_frame=True)
df = iris.frame.copy()

# Match the column names in Datheon's linked Iris CSV.
df = df.rename(columns={
    "sepal length (cm)": "sepal_length",
    "sepal width (cm)": "sepal_width",
    "petal length (cm)": "petal_length",
    "petal width (cm)": "petal_width",
    "target": "species_id"
})
df["species"] = df["species_id"].map(dict(enumerate(iris.target_names)))

FEATURES = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
TARGET = "species"

print("Dataset shape:", df.shape)
display(df.head())
display(df[FEATURES].describe().T)
print("\nMissing values per column:")
display(df.isna().sum().to_frame("missing_values"))
print("\nClass counts:")
display(df[TARGET].value_counts().rename_axis("species").to_frame("count"))


Dataset shape: (150, 6)


,sepal_length,sepal_width,petal_length,petal_width,species_id,species
0,5.1,3.5,1.4,0.2,0,setosa
1,4.9,3.0,1.4,0.2,0,setosa
2,4.7,3.2,1.3,0.2,0,setosa
3,4.6,3.1,1.5,0.2,0,setosa
4,5.0,3.6,1.4,0.2,0,setosa


,count,mean,std,min,25%,50%,75%,max
sepal_length,150.0,5.843333,0.828066,4.3,5.1,5.80,6.4,7.9
sepal_width,150.0,3.057333,0.435866,2.0,2.8,3.00,3.3,4.4
petal_length,150.0,3.758000,1.765298,1.0,1.6,4.35,5.1,6.9
petal_width,150.0,1.199333,0.762238,0.1,0.3,1.30,1.8,2.5



Missing values per column:


,missing_values
sepal_length,0
sepal_width,0
petal_length,0
petal_width,0
species_id,0
species,0



Class counts:


,count
species,
setosa,50
versicolor,50
virginica,50


## 3. Exploratory data analysis

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="species", order=iris.target_names)
plt.title("Class distribution")
plt.xlabel("Species")
plt.ylabel("Number of observations")
plt.tight_layout()
plt.show()

sns.pairplot(df, vars=FEATURES, hue=TARGET, diag_kind="hist", corner=True)
plt.suptitle("Iris feature relationships by species", y=1.02)
plt.show()

plt.figure(figsize=(7, 5))
sns.heatmap(df[FEATURES].corr(), annot=True, cmap="vlag", center=0, fmt=".2f")
plt.title("Feature correlation matrix")
plt.tight_layout()
plt.show()


## 4. Prepare training and test sets

An 80/20 split gives 120 training observations and 30 test observations. `stratify=y` keeps the class distribution balanced across both partitions.

In [ ]:
X = df[FEATURES].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("\nTraining class proportions:")
display(y_train.value_counts(normalize=True).sort_index().to_frame("proportion"))
print("\nTest class proportions:")
display(y_test.value_counts(normalize=True).sort_index().to_frame("proportion"))


## 5. Train both classifiers

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ]),
    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        max_depth=None
    )
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name}: training complete")


## 6. Evaluate model performance

In [ ]:
results = []
predictions = {}

for name, model in models.items():
    y_pred = model.predict(X_test)
    predictions[name] = y_pred
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (macro)": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "F1-score (macro)": f1_score(y_test, y_pred, average="macro", zero_division=0)
    })

metrics_df = pd.DataFrame(results).set_index("Model").sort_values("F1-score (macro)", ascending=False)
display(metrics_df.round(4))

for name, y_pred in predictions.items():
    print(f"\nClassification report — {name}")
    print(classification_report(y_test, y_pred, zero_division=0))


## 7. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(12, 4))
if len(models) == 1:
    axes = [axes]

for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred, labels=list(iris.target_names))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
        xticklabels=iris.target_names, yticklabels=iris.target_names
    )
    ax.set_title(name)
    ax.set_xlabel("Predicted species")
    ax.set_ylabel("Actual species")

plt.tight_layout()
plt.show()


## 8. Compare results visually

In [ ]:
metrics_plot = metrics_df.reset_index().melt(
    id_vars="Model", var_name="Metric", value_name="Score"
)
plt.figure(figsize=(10, 5))
sns.barplot(data=metrics_plot, x="Metric", y="Score", hue="Model")
plt.ylim(0, 1.05)
plt.title("Test-set performance comparison")
plt.ylabel("Score")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## 9. Decision boundary visualization (two selected features)

This visualization is intentionally a **two-feature teaching view** using petal length and petal width. It is not the same as the four-feature model evaluation above; it illustrates how a model separates classes in a 2D projection.

In [ ]:
from matplotlib.colors import ListedColormap

boundary_features = ["petal_length", "petal_width"]
X2 = df[boundary_features]
y_codes = iris.target

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y_codes, test_size=0.20, random_state=RANDOM_STATE, stratify=y_codes
)

boundary_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
])
boundary_model.fit(X2_train, y2_train)

x_min, x_max = X2[boundary_features[0]].min() - 0.5, X2[boundary_features[0]].max() + 0.5
y_min, y_max = X2[boundary_features[1]].min() - 0.5, X2[boundary_features[1]].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))
grid = pd.DataFrame({
    boundary_features[0]: xx.ravel(),
    boundary_features[1]: yy.ravel()
})
zz = boundary_model.predict(grid).reshape(xx.shape)

cmap = ListedColormap(["#dbeafe", "#dcfce7", "#fce7f3"])
plt.figure(figsize=(8, 5))
plt.contourf(xx, yy, zz, alpha=0.65, cmap=cmap)
sns.scatterplot(
    data=df, x=boundary_features[0], y=boundary_features[1],
    hue="species", palette="deep", edgecolor="black", s=55
)
plt.title("Logistic Regression decision regions (two features)")
plt.tight_layout()
plt.show()


## 10. Example prediction on a new observation

In [ ]:
example = pd.DataFrame([{
    "sepal_length": 5.1,
    "sepal_width": 3.5,
    "petal_length": 1.4,
    "petal_width": 0.2
}])

for name, model in models.items():
    print(f"{name} prediction:", model.predict(example)[0])


## 11. Findings, limitations and next steps

After running the notebook, use the metric table and confusion matrices above to write the findings based on the **observed test-set results**. Do not copy assumed accuracy figures.

### Interpretation checklist
- Which model has the stronger macro F1-score on this test split?
- Which species, if any, is confused with another?
- Do the results agree with the feature plots?
- Is the difference meaningful, or could it be affected by the small test set?

### Limitations
- Iris is a small, clean benchmark dataset; performance does not establish performance on messy production data.
- A single train/test split can vary. Cross-validation would provide a more stable estimate.
- The 2D decision-boundary plot uses only two features and should not be confused with the four-feature benchmark.

### Conclusion
The notebook compares two supervised classifiers using a reproducible split and multiple evaluation metrics. The final model choice should be based on the measured results and the intended use case, not accuracy alone.

### Environment
Recommended: Python 3.10+ with `pandas`, `numpy`, `scikit-learn`, `matplotlib`, `seaborn`, and `jupyter`.
Install packages if needed:
```bash
python -m pip install pandas numpy scikit-learn matplotlib seaborn jupyter
```
Run:
```bash
jupyter notebook Task_1_Iris_Classification_Datheon.ipynb
```
